In [2]:
%pip install -r ../requirements.txt

Obtaining file:///home/tmalasda/dev/Cheb_Ar/notebooks (from -r ../requirements.txt (line 2))
ERROR: file:///home/tmalasda/dev/Cheb_Ar/notebooks (from -r ../requirements.txt (line 2)) does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
from cheb_ar.solvers.cheb_ar import *
import time

In [ ]:
# ---------------- parameters (same as your snippet) ----------------
w_a = 25.338776456203686
w_b = 2 * w_a
kappa_b = 5 / 10.4
E_J = 37 * 2 * np.pi
phi_a, phi_b = 0.11, 0.204
epsilon_p = 0.1
g  = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b
kappa_2 = 4 * g**2 / kappa_b
kappa_1 = 0.005 * kappa_2
n_a, n_b = 25, 11
dims = (n_a, n_b)
alpha_sq = 8.5
epsilon_d = 2 * alpha_sq * g2

N = n_a * n_b
T_block = 2 * jnp.pi / w_a
tsave = jnp.array([0.0, T_block])
method = dq.method.Tsit5(rtol=1e-9, atol=1e-10)
opts = dq.Options(assume_hermitian=False)

In [ ]:
from cheb_ar.models.ats import build_ats_hamiltonian
Ham, jump_ops, T_block, params = build_ats_hamiltonian()
solver = ChebAr(Ham, jump_ops, T_block, dims=dims)
m_arnoldi_0 = 60
x0 = solver.make_x0(seed=0)
#_, _, ritz_vals = solver.first_estimation(x0, m_arnoldi=m_arnoldi_0)

In [ ]:
from cheb_ar.models.ats import build_ats_hamiltonian_interaction

In [ ]:
H_I, jump_ops_I, jump_ops_LdL_I, output_phase, V, T_block, params = build_ats_hamiltonian_interaction()
solver = ChebAr(H_I, jump_ops_I, T_block, jump_ops_LdL=jump_ops_LdL_I, dims=dims, output_phase=output_phase)
_, _, ritz_vals = solver.first_estimation(x0, m_arnoldi=m_arnoldi_0)

In [ ]:
margin = 1e-2
solver.setup_chebyshev(ritz_vals, margin=margin)
warm_start = False

In [ ]:
m_arnoldi = 120
Q, H, mu_list = solver.arnoldi_hessenberg(x0, solver.chebyshev_filter, m_arnoldi, warm_start = warm_start)

In [ ]:
rate_bf = solver.rate_from_mu(mu_list[-1])
x_ritz, _ = solver.ritz_vector(Q, H, m_arnoldi)
res = solver.residual_check(x_ritz)

In [ ]:
rate_bf

In [ ]:
rho_lab   = V @ x_ritz.reshape(N, N) @ V.conj().T

In [ ]:
rho_lab_dq = dq.asqarray(rho_lab, dims=dims)

In [ ]:
dq.plot.wigner(dq.ptrace(rho_lab_dq,0))